### Part 2 


In [ ]:
# Import neccessary libraries
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.metrics import mean_squared_error, r2_score


df = pd.read_csv("../data/processed/berlin_2024_Q1_clean.csv")
df.shape


(9264, 13)

After data cleaning and the removal of invalid or missing price observations, the final analytical sample contains 9,264 listings. This reduction reflects standard preprocessing practices and ensures that the models are trained on valid and reliable observations.

In [ ]:
# Convert EVERYTHING to numpy arrays
X_train_np = np.asarray(X_train, dtype=float)
X_test_np = np.asarray(X_test, dtype=float)
y_train_np = np.asarray(y_train, dtype=float)
y_test_np = np.asarray(y_test, dtype=float)

# Add constant manually
X_train_np = sm.add_constant(X_train_np)
X_test_np = sm.add_constant(X_test_np)

# Fit OLS
ols_model = sm.OLS(y_train_np, X_train_np).fit()

print(ols_model.summary())


                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.421
Model:                            OLS   Adj. R-squared:                  0.420
Method:                 Least Squares   F-statistic:                     414.5
Date:                Mon, 02 Feb 2026   Prob (F-statistic):               0.00
Time:                        13:48:33   Log-Likelihood:                -5621.0
No. Observations:                7411   AIC:                         1.127e+04
Df Residuals:                    7397   BIC:                         1.137e+04
Df Model:                          13                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const        -35.3430      9.359     -3.776      0.0

The OLS model explains approximately 42% of the variation in log Airbnb prices, which is consistent with findings in the housing and platform pricing literature. Key structural and quality-related variables are statistically significant and economically meaningful. The presence of multicollinearity motivates the use of regularized models in the next step.

In [ ]:
# Lasso Regression
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lasso = LassoCV(cv=5, random_state=42)
lasso.fit(X_train_scaled, y_train)

y_pred_lasso = lasso.predict(X_test_scaled)

rmse_lasso = np.sqrt(mean_squared_error(y_test, y_pred_lasso))
r2_lasso = r2_score(y_test, y_pred_lasso)

rmse_lasso, r2_lasso



(np.float64(0.5113168405059069), 0.4207489228630794)

The LASSO model achieves a similar level of predictive performance compared to OLS, with an R² of approximately 0.42. This suggests that regularization does not substantially reduce explanatory power, while providing a more parsimonious model by shrinking less relevant coefficients toward zero. As such, LASSO offers a favorable trade-off between predictive accuracy and model simplicity.

In [ ]:
# Number of selected features
np.sum(lasso.coef_ != 0)

lasso_coefs = pd.Series(lasso.coef_, index=X.columns)
lasso_coefs[lasso_coefs != 0].sort_values(key=abs, ascending=False)



accommodates              0.232201
room_type_Private room   -0.183723
room_type_Shared room    -0.113330
room_type_Hotel room      0.094842
kitchen                  -0.087482
air_conditioning          0.085414
bedrooms                  0.066674
bathrooms                 0.062482
review_scores_rating      0.053552
number_of_reviews        -0.036185
latitude                  0.025985
wifi                      0.007662
dtype: float64

LASSO identifies accommodation capacity and room type as the primary drivers of Airbnb prices. Entire-home listings and higher guest capacity significantly increase prices, while private and shared rooms are associated with substantial discounts. Amenities such as air conditioning and kitchens contribute positively but with smaller magnitudes. Review-related variables play a secondary role once structural characteristics are controlled for. Overall, the results are economically intuitive and consistent with platform pricing mechanisms.